In [ ]:
"""
create_changed_inference_data.py

Purpose
-------
Create a changed version of the reserved inference validation dataset
by swapping two feature columns.

Input
-----
data/inference_validation/hmda_inference_validation.csv

Output
------
data/inference_validation/hmda_inference_validation_changed.csv
outputs/changed_feature_summary.txt

Main logic
----------
1. Load the reserved 10,000-row inference validation dataset.
2. Select two feature columns to swap.
3. Create a changed dataset by swapping the two feature columns.
4. Save the changed dataset.
5. Save a short summary of the feature change.

"""

import pandas as pd
import numpy as np

from pathlib import Path


# ============================================================
# 1. Project paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

INPUT_PATH = (
    PROJECT_ROOT / "data" / "inference_validation" / "hmda_inference_validation.csv"
)

CHANGED_OUTPUT_PATH = (
    PROJECT_ROOT / "data" / "inference_validation" / "hmda_inference_validation_changed.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
SUMMARY_PATH = OUTPUT_DIR / "changed_feature_summary.txt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHANGED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Settings
# ============================================================

TARGET = "loan_approved"

# Preferred feature pairs to swap.
# The script will use the first pair that exists in your dataset.
PREFERRED_SWAP_PAIRS = [
    ("loan_amount_log", "property_value_log"),
    ("loan_amount", "property_value"),
    ("income_log", "loan_to_value_ratio"),
    ("income", "loan_to_value_ratio"),
]


# ============================================================
# 3. Helper functions
# ============================================================

def select_swap_features(df):
    """
    Select two feature columns to swap.

    The function first tries preferred feature pairs.
    If none of them exist, it falls back to the first two numeric feature columns.
    """

    for feature_a, feature_b in PREFERRED_SWAP_PAIRS:
        if feature_a in df.columns and feature_b in df.columns:
            return feature_a, feature_b

    numeric_features = [
        col
        for col in df.select_dtypes(include=[np.number]).columns
        if col != TARGET
    ]

    if len(numeric_features) < 2:
        raise ValueError(
            "Unable to find two numeric feature columns for swapping."
        )

    return numeric_features[0], numeric_features[1]


def create_changed_data_by_swapping(df, feature_a, feature_b):
    """
    Create a changed dataset by swapping two feature columns.
    """

    changed_df = df.copy()

    temp_values = changed_df[feature_a].copy()
    changed_df[feature_a] = changed_df[feature_b]
    changed_df[feature_b] = temp_values

    return changed_df


def summarize_feature_change(original_df, changed_df, feature_a, feature_b):
    """
    Create a summary showing how the two selected features changed.
    """

    summary = {
        "feature_a": feature_a,
        "feature_b": feature_b,
        "rows": len(original_df),

        f"original_{feature_a}_mean": original_df[feature_a].mean(),
        f"changed_{feature_a}_mean": changed_df[feature_a].mean(),

        f"original_{feature_b}_mean": original_df[feature_b].mean(),
        f"changed_{feature_b}_mean": changed_df[feature_b].mean(),

        f"original_{feature_a}_first_value": original_df[feature_a].iloc[0],
        f"changed_{feature_a}_first_value": changed_df[feature_a].iloc[0],

        f"original_{feature_b}_first_value": original_df[feature_b].iloc[0],
        f"changed_{feature_b}_first_value": changed_df[feature_b].iloc[0],
    }

    return summary


# ============================================================
# 4. Main feature changing pipeline
# ============================================================

def main() -> None:

    # ------------------------------------------------------------
    # Load inference validation data
    # ------------------------------------------------------------

    if not INPUT_PATH.exists():
        raise FileNotFoundError(
            f"Inference validation data not found: {INPUT_PATH}\n"
            "Please run the feature preparation script first."
        )

    df = pd.read_csv(INPUT_PATH, low_memory=False)

    print("Original inference validation data shape:", df.shape)

    if TARGET not in df.columns:
        raise ValueError(f"Target column not found: {TARGET}")

    # ------------------------------------------------------------
    # Select two features to swap
    # ------------------------------------------------------------

    feature_a, feature_b = select_swap_features(df)

    print()
    print("Selected features for swapping:")
    print("Feature A:", feature_a)
    print("Feature B:", feature_b)

    # ------------------------------------------------------------
    # Create changed dataset
    # ------------------------------------------------------------

    changed_df = create_changed_data_by_swapping(
        df=df,
        feature_a=feature_a,
        feature_b=feature_b,
    )

    # ------------------------------------------------------------
    # Save changed dataset
    # ------------------------------------------------------------

    changed_df.to_csv(
        CHANGED_OUTPUT_PATH,
        index=False,
    )

    print()
    print("Changed inference validation data saved to:")
    print(CHANGED_OUTPUT_PATH)

    # ------------------------------------------------------------
    # Save summary
    # ------------------------------------------------------------

    summary = summarize_feature_change(
        original_df=df,
        changed_df=changed_df,
        feature_a=feature_a,
        feature_b=feature_b,
    )

    with open(SUMMARY_PATH, "w") as f:
        f.write("Changed Feature Data Summary\n")
        f.write("============================\n\n")

        f.write("Script purpose:\n")
        f.write(
            "This script creates a changed version of the reserved "
            "inference validation dataset by swapping two feature columns.\n\n"
        )

        f.write(f"Input path: {INPUT_PATH}\n")
        f.write(f"Changed output path: {CHANGED_OUTPUT_PATH}\n")
        f.write(f"Rows: {len(df)}\n")
        f.write(f"Target column: {TARGET}\n\n")

        f.write("Changed features:\n")
        f.write(f"Feature A: {feature_a}\n")
        f.write(f"Feature B: {feature_b}\n")
        f.write("Change method: swap two feature columns\n\n")

        f.write("Summary statistics:\n")
        for key, value in summary.items():
            f.write(f"{key}: {value}\n")

    print()
    print("Changed feature summary saved to:")
    print(SUMMARY_PATH)

    print()
    print("Changed feature data creation completed successfully.")


if __name__ == "__main__":
    main()

Original inference validation data shape: (10000, 30)

Selected features for swapping:
Feature A: loan_amount
Feature B: property_value

Changed inference validation data saved to:
/Users/panshen/Desktop/MLOps_Final/data/inference_validation/hmda_inference_validation_changed.csv

Changed feature summary saved to:
/Users/panshen/Desktop/MLOps_Final/outputs/changed_feature_summary.txt

Changed feature data creation completed successfully.
